# 付録：多項ロジット確率式の導出

多項ロジット（MNL）の確率式

$$P_{ij} = \frac{\exp(V_{ij})}{\displaystyle\sum_{k=1}^{J}\exp(V_{ik})}$$

が、ランダム効用モデルの仮定からどのように導かれるかを整理する。


## 前提設定

消費者 $i$ が選択肢 $j$ から得る効用を **確定的効用** $V_{ij}$ と **確率的誤差項** $\varepsilon_{ij}$ に分解する：

$$U_{ij} = V_{ij} + \varepsilon_{ij}$$

効用最大化原理より、消費者 $i$ が選択肢 $j$ を選ぶ確率は：

$$P_{ij} = P(U_{ij} > U_{ik},\; \forall k \neq j)$$

**仮定**

- $\varepsilon_{ij}$ は選択肢 $j$ 間で **独立同一** の標準ガンベル分布（位置0、尺度1）に従う
- ガンベル分布の CDF：$F(x) = \exp(-\exp(-x))$
- ガンベル分布の PDF：$f(x) = \exp(-x)\exp(-\exp(-x))$
- 連続分布なので効用が同値になる確率は0であり、$>$ と $\ge$ の違いは選択確率に影響しない


## Step 1：$\varepsilon_{ij}$ を固定した条件付き確率

$\varepsilon_{ij}=e$ に固定する。選択肢 $j$ が選ばれるには、すべての $k \neq j$ について

$$V_{ij}+e > V_{ik}+\varepsilon_{ik}$$

が成り立つ必要がある。これは

$$\varepsilon_{ik} < e + V_{ij} - V_{ik}$$

と同値である。誤差項が選択肢間で独立なので、条件付き確率は積に分解できる：

$$P(j\text{を選択}\mid \varepsilon_{ij}=e)
= \prod_{k \neq j} F(e+V_{ij}-V_{ik})$$

ガンベル分布の CDF を代入すると、

$$
\begin{aligned}
P(j\text{を選択}\mid \varepsilon_{ij}=e)
&= \prod_{k \neq j} \exp\{-\exp[-(e+V_{ij}-V_{ik})]\} \\
&= \exp\left[-\exp(-e)\sum_{k \neq j}\frac{\exp(V_{ik})}{\exp(V_{ij})}\right].
\end{aligned}
$$

ここで

$$B = \sum_{k \neq j}\frac{\exp(V_{ik})}{\exp(V_{ij})}$$

とおくと、

$$P(j\text{を選択}\mid \varepsilon_{ij}=e) = \exp[-B\exp(-e)]$$

と書ける。


## Step 2：$\varepsilon_{ij}$ について積分（全確率の法則）

$\varepsilon_{ij}$ の固定を解いて、$e$ の全域で積分する：

$$
\begin{aligned}
P_{ij}
&= \int_{-\infty}^{\infty} P(j\text{を選択}\mid \varepsilon_{ij}=e) f(e)\,de \\
&= \int_{-\infty}^{\infty} \exp[-B\exp(-e)]\,\exp(-e)\exp[-\exp(-e)]\,de \\
&= \int_{-\infty}^{\infty} \exp(-e)\exp[-(B+1)\exp(-e)]\,de.
\end{aligned}
$$

ここで

$$
B+1
= \sum_{k \neq j}\frac{\exp(V_{ik})}{\exp(V_{ij})} + 1
= \frac{\sum_{k=1}^{J}\exp(V_{ik})}{\exp(V_{ij})}
$$

である。最後の等号では、$+1$ を $\exp(V_{ij})/\exp(V_{ij})$ と見て、$j$ 自身の項を総和に加えている。


## Step 3：置換積分

$$C = B+1 = \frac{\sum_{k=1}^{J}\exp(V_{ik})}{\exp(V_{ij})}$$

とおくと、積分は

$$P_{ij}=\int_{-\infty}^{\infty}\exp(-e)\exp[-C\exp(-e)]\,de$$

となる。ここで $t=C\exp(-e)$ と置換する。

$$dt = -C\exp(-e)\,de \quad\Rightarrow\quad \exp(-e)\,de = -\frac{dt}{C}$$

積分範囲は、$e\to -\infty$ で $t\to \infty$、$e\to \infty$ で $t\to 0$ である。したがって、

$$
\begin{aligned}
P_{ij}
&= \int_{\infty}^{0} e^{-t}\left(-\frac{dt}{C}\right) \\
&= \frac{1}{C}\int_{0}^{\infty}e^{-t}\,dt \\
&= \frac{1}{C}.
\end{aligned}
$$

$C$ を戻すと、

$$
P_{ij}
= \frac{\exp(V_{ij})}{\sum_{k=1}^{J}\exp(V_{ik})}
$$

が得られる。


## 結果

$$\boxed{P_{ij} = \frac{\exp(V_{ij})}{\displaystyle\sum_{k=1}^{J}\exp(V_{ik})}}$$

**各ステップで使った仮定のまとめ：**

| ステップ | 操作 | 使った仮定 |
|---|---|---|
| Step 1 | 積の形に分解 | 誤差項の**選択肢間独立性** |
| Step 1 | CDF $F$ で表現 | 誤差項が**同一分布**（ガンベル分布） |
| Step 2 | $f(e)$ との積を積分 | ガンベル分布の**PDF** |
| Step 3 | 置換積分で閉じた解に | $\int_0^\infty e^{-t}dt = 1$ |

> **なぜガンベル分布でないとうまくいかないのか？**
> 正規分布を仮定した場合（プロビットモデル）は Step 1 の積がこの形に整理できず、Step 3 の置換積分による閉じた解が得られない。そのため数値積分が必要になる。


## 数値確認：モンテカルロシミュレーションによる検証

ガンベル分布から乱数を生成し、選択確率を直接シミュレーションして解析解と一致するか確認する。


In [19]:
import numpy as np

rng = np.random.default_rng(42)
N = 1_000_000  # シミュレーション回数

# 確定的効用（3選択肢）
V = np.array([1.0, 0.5, -0.5])
J = len(V)

# ガンベル分布乱数：U(0,1) → Gumbel 変換
u = rng.uniform(0, 1, size=(N, J))
eps = -np.log(-np.log(u))  # Gumbel サンプル

# 効用 U_ij = V_j + eps_j
U = V + eps  # ブロードキャスト

# 各試行で最大効用の選択肢を選ぶ
choices = np.argmax(U, axis=1)

# シミュレーション選択確率
sim_prob = np.bincount(choices, minlength=J) / N

# 解析解（MNL）
exp_V = np.exp(V)
mnl_prob = exp_V / exp_V.sum()

print("選択肢 j |  解析解 P_ij  | シミュレーション | 差")
print("-" * 55)
for j in range(J):
    print(
        f"  j={j}   |   {mnl_prob[j]:.6f}   |   {sim_prob[j]:.6f}    | {abs(mnl_prob[j] - sim_prob[j]):.6f}"
    )

選択肢 j |  解析解 P_ij  | シミュレーション | 差
-------------------------------------------------------
  j=0   |   0.546549   |   0.546925    | 0.000376
  j=1   |   0.331499   |   0.331277    | 0.000222
  j=2   |   0.121952   |   0.121798    | 0.000154
